In [ ]:
import cv2
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from skimage.feature import greycomatrix,greycoprops

In [ ]:
img = cv2.imread('./test_chun1.jpeg',-1)
img2 = cv2.imread('./test_chun2.jpeg',-1)
img3 = cv2.imread('./test_jin.jpg',0)
scaler = MinMaxScaler()

In [ ]:
hist = cv2.calcHist([img],[0],None,[256],[0,256])
hist2 = cv2.calcHist([img2],[0],None,[256],[0,256])
hist3 = cv2.calcHist([img3],[0],None,[256],[0,256])
hist = cv2.normalize(hist,hist).flatten()
hist2 = cv2.normalize(hist2,hist2).flatten()
hist3 = cv2.normalize(hist3,hist3).flatten()

In [ ]:
x = cv2.compareHist(hist,hist2,cv2.HISTCMP_INTERSECT)
x

In [ ]:
x2 = cv2.compareHist(hist,hist3,cv2.HISTCMP_INTERSECT)
x2

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
plt.hist(img.ravel(),256,[0,256]);
plt.show()

In [ ]:
plt.hist(img2.ravel(),256,[0,256])
plt.show()

In [ ]:
plt.hist(img3.ravel(),256,[0,256])
plt.show()

In [ ]:
glcm= greycomatrix(img,[1],[0],symmetric = True,normed = True)
glcm2 = greycomatrix(img2,[1],[0],symmetric = True,normed = True)
glcm3 = greycomatrix(img3,[1],[0],symmetric = True,normed = True)

In [ ]:
np.sum((glcm - glcm2)**2)**0.5

In [ ]:
np.sum((glcm2 - glcm3)**2)**0.5

In [ ]:
np.sum((glcm - glcm3)**2)**0.5

In [ ]:
def CombineSimilarity(img,img2):
    hist = cv2.calcHist([img],[0],None,[256],[0,256])
    hist2 = [ cv2.calcHist([x],[0],None,[256],[0,256]) for x in img2 ]
    hist = cv2.normalize(hist,hist).flatten()
    hist2 = [ cv2.normalize(x,x).flatten() for x in hist2]
    HisSimilarity = np.array([ cv2.compareHist(hist,x,cv2.HISTCMP_INTERSECT) for x in hist2])
    
    glcm= greycomatrix(cv2.cvtColor(img,cv2.COLOR_RGB2GRAY),[1],[0],symmetric = True,normed = True)
    glcm2 = [ greycomatrix(cv2.cvtColor(x,cv2.COLOR_RGB2GRAY),[1],[0],symmetric = True,normed = True) for x in img2]
    GLCMSimilarity = np.sum((glcm - glcm2)**2)**0.5
    return 0.7*HisSimilarity + 0.3*GLCMSimilarity
    

In [ ]:
def CombineSimilarity_single(img,img2):
    hist = cv2.calcHist([img],[0],None,[256],[0,256])
    hist2 = cv2.calcHist([img2],[0],None,[256],[0,256])
    hist = cv2.normalize(hist,hist).flatten()
    hist2 = cv2.normalize(hist2,hist2).flatten()
    HisSimilarity = cv2.compareHist(hist,hist2,cv2.HISTCMP_INTERSECT)
    print(HisSimilarity)
    
    glcm= greycomatrix(img,[1],[0],symmetric = True,normed = True)
    glcm2 = greycomatrix(img2,[1],[0],symmetric = True,normed = True)
    GLCMSimilarity = np.sum((glcm - glcm2)**2)**0.5
    print(GLCMSimilarity)
    return 0.7*HisSimilarity + 0.3*GLCMSimilarity

In [ ]:
CombineSimilarity_single(img,img2)

In [ ]:
CombineSimilarity_single(img,img3)

In [ ]:
CombineSimilarity_single(img,img)

# video clustering

In [ ]:
vidcap = cv2.VideoCapture("./test_video.mp4")

test_image_set = []
count = 0

while(vidcap.isOpened):
    ret, image = vidcap.read()
    print(ret)
    if ret:
        test_image_set.append(image)
        count +=1
        print(f"frame {count}")
    else:
            break
vidcap.release()

In [ ]:
class Clustering:
    def __init__(self,train_x):
        self.numCluster = 1
        self.Cluster = []
        self.centroid = []
        self.a = 6.0
        self.learn(train_x)
    def learn(self,train_x):
        self.Cluster.append([train_x[0]])
        self.centroid.append(train_x[0])
        for idx,x in enumerate(train_x[1:]):
            s = np.max(CombineSimilarity(x,self.centroid))
            idx = np.argmax(CombineSimilarity(x,self.centroid))
            print(f'{s} ')
            if(s > self.a):
                print(f'Cluster idx : {idx}')
                self.Cluster[idx].append(x)
                numFrame = len(self.Cluster[idx])
                self.centroid[idx] = self.Cluster[idx][numFrame//2]
            else:
                print(f'Cluster idx : {self.numCluster-1}')
                self.Cluster.append([x])
                self.numCluster+=1
                self.centroid.append(x)
                
            
                
            

In [ ]:
cl = Clustering(test_image_set)

In [ ]:
count = 0
for i in range(len(cl.centroid)):
    if len(cl.Cluster[i]) >= len(test_image_set)*0.1:
        print(i)
        cv2.imwrite('test_image_' + str(count) + '.jpg',cl.centroid[i])
        count += 1